# Spam Review Detection Model
Training Random Forest and SVM models for spam detection

In [ ]:
!pip install scikit-learn pandas numpy nltk joblib -q

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import re

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

In [ ]:
# Create training data
data = {
    'review': [
        'Excellent quality product',
        'Very good experience',
        'Satisfied with purchase',
        'Great value for money',
        'BUY NOW LIMITED OFFER!!!',
        'Click here for discount code',
        'Best seller only at our store',
        'Free shipping now!!!',
    ],
    'is_spam': [0, 0, 0, 0, 1, 1, 1, 1]  # 0=Not Spam, 1=Spam
}

df = pd.DataFrame(data)
print('Spam Distribution:')
print(df['is_spam'].value_counts())

In [ ]:
# Preprocessing
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [stemmer.stem(token) for token in tokens if token not in stop_words]
    return ' '.join(tokens)

df['review_processed'] = df['review'].apply(preprocess_text)

In [ ]:
# Feature Extraction
vectorizer = TfidfVectorizer(max_features=5000, min_df=1)
X = vectorizer.fit_transform(df['review_processed'])
y = df['is_spam']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Train Random Forest
print('Training Random Forest...')
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print('\n=== Random Forest Performance ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_rf):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_rf):.4f}')
print(f'F1-Score: {f1_score(y_test, y_pred_rf):.4f}')

In [ ]:
# Train SVM
print('Training SVM...')
svm_model = SVC(kernel='rbf', probability=True, random_state=42)
svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)

print('\n=== SVM Performance ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_svm):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_svm):.4f}')
print(f'F1-Score: {f1_score(y_test, y_pred_svm):.4f}')

In [ ]:
# Save models
import os
os.makedirs('/content/models', exist_ok=True)

joblib.dump(rf_model, '/content/models/spam_model.pkl')
print('✓ Spam detection model saved')